# Versjonskontroll med Git

[Git](https://git-scm.com/) er et **versjonskontrollsystem** som holder orden på ulike versjoner av filene våre mens vi utvikler. Fordeler:

* Vi kan angre og gå tilbake til tidligere versjoner.
* Flere kan jobbe med de samme filene samtidig (KM09).
* Vi kan jobbe med flere versjoner parallelt.
* Git er gratis og har åpen kildekode.

Git kan brukes uavhengig av [GitHub](https://github.com), som er en skytjeneste bygget på Git med mange nyttige funksjoner for utviklingsprosjekter. I juni 2022 hadde GitHub over 83 millioner utviklere og over 200 millioner *repositories*.

Git er **distribuert**: alle kan laste ned og jobbe med filene lokalt, og så lastes endringene opp og flettes inn i et felles lager. I denne runden bruker vi Git **lokalt**, bare for oss selv. Samarbeid med GitHub kommer i 4.7.

> I denne notebooken kjører vi Git-kommandoer fra Python, slik at du kan prøve alt direkte. Det krever at Git er installert: <https://git-scm.com/downloads>. Til vanlig skriver du kommandoene i terminalen (PowerShell) eller bruker VS Code.

## 1 Arbeidsområde, staging og repo

| Engelsk | Norsk | Forklaring |
|---|---|---|
| *working directory* | arbeidsområdet | Mappen der vi jobber med filene. Filer som bare finnes her, er *untracked*. |
| *staging area* | staging-området | Et mellomlager. Filer legges hit med `git add` før de lagres i repoet. |
| *repository* | repo | Lageret med hele historikken, i den skjulte mappen `.git`. |

```mermaid
sequenceDiagram
    participant W as Arbeidsområdet
    participant S as Staging
    participant R as Repo (.git)
    W->>S: git add
    S->>R: git commit
    R->>W: git restore
```

Vi bestemmer selv hvilke filer Git skal følge med på (`git add`). `git commit` lagrer en **versjon** av filene i staging-området i repoet. Denne totrinnsprosessen gjentar vi hver gang vi vil lagre en versjon.
`git commit -am "melding"` gjør begge deler på én gang for filer Git allerede følger (ikke nye filer).

## 2 Git-ordliste

| Kommando | Norsk | Forklaring |
|---|---|---|
| `git init` | – | lager et nytt repo i mappen |
| `git status` | – | viser hva som er endret, lagt til osv. |
| `git add` | adde | legger filer i staging-området |
| `git commit` | committe | lagrer filene i staging-området i repoet |
| `git log` | – | viser historikken |
| `git diff` | – | viser forskjeller mellom versjoner |
| `git restore` | – | henter filer fra staging-området eller repoet til arbeidsområdet |
| *branch* | gren | versjonene lagres langs en hovedlinje (`master`/`main`). Nye funksjoner utvikles gjerne i en egen gren |
| `git branch` | branche | viser eller lager grener |
| `git switch` | switche | bytter gren (`checkout` er en eldre, mer generell kommando) |
| `git merge` | merge | fletter endringene fra en gren inn i en annen |

## 3 Kom i gang

Sjekk at Git er installert, og sett brukernavn og e-post. Det gjøres én gang per maskin:

```
git --version
git config --global user.name "Ditt Navn"
git config --global user.email "din@epost.no"
git config --global --list
```

Cellen under lager en hjelpefunksjon `git(...)` som kjører kommandoer i en egen øvingsmappe `git_ovelse/A`:

In [ ]:
import subprocess, shutil, os
from pathlib import Path

repo = Path('git_ovelse') / 'A'
if repo.exists():
    shutil.rmtree(repo, onerror=lambda f, p, e: (os.chmod(p, 0o700), f(p)))   # start på nytt hver gang
repo.mkdir(parents=True)

def git(*argumenter):
    """Kjør en git-kommando i øvingsmappen og vis utskriften."""
    print('$ git', ' '.join(argumenter))
    r = subprocess.run(['git', *argumenter], cwd=repo, capture_output=True, text=True)
    print(r.stdout + r.stderr)

def skriv(tekst, filnavn='tall.txt'):
    """Skriv tekst til en fil i øvingsmappen (i stedet for notepad)."""
    (repo / filnavn).write_text(tekst, encoding='utf-8')

def vis(filnavn='tall.txt'):
    print((repo / filnavn).read_text(encoding='utf-8'))

git('--version')

## 4 Øvelse: Git fra kommandolinjen

Følg stegene og legg merke til hva `git status` sier underveis. (I PowerShell skriver du `git init A`, `cd A`, `notepad tall.txt` osv.)

In [ ]:
git('init', '-b', 'master')
git('config', 'user.name', 'Elev')            # bare for denne øvelsen
git('config', 'user.email', 'elev@skole.no')
print('.git finnes:', (repo / '.git').exists())
git('status')

Lag `tall.txt` med fire linjer. Filen er nå *untracked*:

In [ ]:
skriv('en\nto\ntre\nfire\n')
vis()
git('status')

Legg filen i staging-området og lagre den første versjonen:

In [ ]:
git('add', 'tall.txt')
git('status')
git('commit', '-m', 'Første import av tall.txt')
git('status')

Endre filen så den har seks linjer. `git diff` viser forskjellen mellom arbeidsområdet og staging-området:

In [ ]:
skriv('en\nto\ntre\nfire\nfem\nseks\n')
git('diff')

In [ ]:
git('add', 'tall.txt')
git('status')
git('diff')              # tom – arbeidsområdet er lik staging
git('diff', 'HEAD')      # forskjellen mot siste versjon i repoet
git('commit', '-m', '6 tall hvorav 2 nye')
git('log', '--oneline')

### Hente fram en gammel versjon

`git restore --source master~1 tall.txt` henter filen slik den var én versjon tilbake (`~n` = n versjoner bakover). Så lagrer vi den som en *ny* versjon, slik at historikken beholdes.

In [ ]:
git('restore', '--source', 'master~1', 'tall.txt')
vis()
git('commit', '-am', 'Restore 1. versjon med 4 tall')

### Grener

Vi lager grenen `engelsk`, bytter til den og endrer filen. Hovedgrenen påvirkes ikke.

In [ ]:
git('branch')
git('branch', 'engelsk')
git('switch', 'engelsk')
git('branch')                # * viser hvilken gren vi er på

skriv('en\nto\ntre\nfour\nfive\nsix\n')
git('commit', '-am', '6 tall, de 3 siste på engelsk')

Tilbake på `master` har filen fortsatt fire tall. Vi legger til «fem» og lagrer:

In [ ]:
git('switch', 'master')
vis()
skriv('en\nto\ntre\nfire\nfem\n')
git('commit', '-am', '5 tall på norsk')

### Merge og konflikter

Nå er `tall.txt` endret på de samme linjene i begge grenene. Når vi fletter, får vi en **konflikt** som Git ikke kan løse selv:

In [ ]:
git('merge', 'engelsk')
vis()

Git har markert konflikten i filen:

```
<<<<<<< HEAD        (det som står i master)
...
=======
...
>>>>>>> engelsk     (det som kommer fra engelsk)
```

Vi løser den ved å redigere filen slik vi vil ha den, fjerne markeringene og committe:

In [ ]:
skriv('en\nto\ntre\nfire\nfem\nsix\n')
git('commit', '-am', 'Merge branch engelsk')
git('merge', 'engelsk')          # nå er alt flettet
git('log', '--oneline', '--graph', '--all')

## 5 Git i VS Code

* Klikk **Source Control** (**Ctrl+Shift+G**) i verktøylinja og *Initialize Repository* for å lage et repo.
* Klikk **+** (*Stage Changes*) ved filnavnet for å adde. Skriv en melding og klikk ✓ *Commit*.
* Klikk på et filnavn for å se endringene siden forrige commit.
* Utvidelsen **GitLens** gjør det lett å bla i tidligere versjoner, sammenligne to versjoner (*Select for Compare* / *Compare with Selected*) og hente fram en gammel versjon (*Restore (Checkout)*).
* Grener lages og byttes nede til venstre i statuslinja (der det står `master`). Konflikter kan løses i *Merge Editor*.
* *Show Commit Graph* viser historikken grafisk.

Se videoen [Using Git with Visual Studio Code](https://code.visualstudio.com/docs/sourcecontrol/overview) og [How to Undo Changes in Git](https://www.atlassian.com/git/tutorials/undoing-changes) (`git reset` og `git restore`).

Mer: [Git Reference](https://git-scm.com/docs), [Git Tutorial (W3Schools)](https://www.w3schools.com/git/).

> **Tips:** Notebooks (`.ipynb`) lagrer også utdata. Tøm utdataene (*Clear All Outputs*) før du committer, så blir `git diff` mye lettere å lese.

## 6 Oppgaver

### Oppgave 1 – Git i terminalen
Gjør øvelsen i del 4 selv i PowerShell: lag `C:\repos`, kjør `git init A`, og bruk `notepad tall.txt` og `type tall.txt` i stedet for hjelpefunksjonene.

### Oppgave 2 – Git i VS Code
Gjør den samme øvelsen i en ny mappe `C:\repos\B`, men bare med VS Code (Source Control og GitLens):

1. Åpne mappen med *File → Open Folder* og velg *Initialize Repository*.
2. Lag `tall.txt` med fire linjer, adde og commit «Første import av tall.txt».
3. Legg til to linjer, se forskjellen, og commit «6 tall hvorav 2 nye».
4. Hent fram første versjon med *Restore (Checkout)* og commit.
5. Lag grenen `engelsk`, endre de tre siste tallene til engelsk og commit.
6. Bytt til `master`, legg til «fem» og commit.
7. Merge `engelsk` inn i `master`, løs konflikten i Merge Editor og commit.
8. Se på historikken med *Show Commit Graph*.

### Oppgave 3 – Fiff opp terminalen (valgfri)
* Installer siste versjon av [PowerShell](https://aka.ms/powershell) og [Windows Terminal](https://aka.ms/terminal) fra Microsoft Store. Sjekk versjonen med `$PSVersionTable`.
* Installer en [Nerd Font](https://www.nerdfonts.com/) (for eksempel *Hack Nerd Font*) og [Oh My Posh](https://ohmyposh.dev/).
* Installer modulene `posh-git` og `Terminal-Icons` fra [PowerShell Gallery](https://www.powershellgallery.com/) med `Install-Module`.
* Sett skrifta i VS Code-terminalen (*Terminal › Integrated: Font Family*) til `Hack Nerd Font Mono`.

Nå viser prompten hvilken gren du er på og om det finnes endringer.